In [3]:
import os, io, zipfile, requests
import pandas as pd
from pathlib import Path

In [7]:
BASE483  = Path("/home/scratch/jtoniolo/483") 
CSV_PATH = "Excessive Rainfall Outlook Days - ML Training.csv"
DATE_COL = "date" 

## Connecting to the location where the ERO archives are stored and downloading the dates listed in our csv

Disclaimer: I had help from ChatGPT as I am not very familiar with automatically downloading from an https location. Before this notebook I did a lot of testing on a singular day and it worked fine for me, I wanted to ensure I did not mess anything up with bulk downloading so I added a lot of print statements to check my progress and whatnot.

In [8]:
ZIP_DIR = BASE483 / "ero_zips_94e09"
ZIP_DIR.mkdir(parents=True, exist_ok=True)

base_urls = [
    "https://www.wpc.ncep.noaa.gov/archives/ero",
    "http://www.wpc.ncep.noaa.gov/archives/ero",
]
headers = {"User-Agent": "Mozilla/5.0"}

df = pd.read_csv(CSV_PATH)

if DATE_COL not in df.columns:
    DATE_COL = df.columns[0]
    print("Using DATE_COL =", DATE_COL)

ymds = (pd.to_datetime(df[DATE_COL], errors="coerce")
          .dt.strftime("%Y%m%d")
          .dropna()
          .drop_duplicates()
          .tolist())

# two quick checks I made to ensure I was doing it correctly
print("Unique dates to download:", len(ymds))
print("First 5:", ymds[:5])

for ymd in ymds:
    fname = f"shp_94e_{ymd}09.zip"
    zip_path = ZIP_DIR / fname

    # skip if already downloaded... I had a couple in the folder manually from testing
    if zip_path.exists() and zip_path.stat().st_size > 1000:
        continue

    ok = False
    for base in base_urls:
        url = f"{base}/{ymd}/{fname}"
        try:
            r = requests.get(url, headers=headers, timeout=60)
            if r.status_code == 200 and len(r.content) > 1000:
                zip_path.write_bytes(r.content)
                print("OK:", ymd)
                ok = True
                break
        except Exception as e:
            pass

    if not ok:
        print("MISS:", ymd)

Unique dates to download: 999
First 5: ['20250129', '20250130', '20250211', '20250212', '20250215']
OK: 20250129
OK: 20250130
OK: 20250211
OK: 20250212
OK: 20250215
OK: 20250304
OK: 20250309
OK: 20250315
OK: 20250326
OK: 20250327
OK: 20250328
OK: 20250330
OK: 20250331
OK: 20250402
OK: 20250403
OK: 20250404
OK: 20250405
OK: 20250406
OK: 20250407
OK: 20250418
OK: 20250419
OK: 20250420
OK: 20250424
OK: 20250425
OK: 20250426
OK: 20250427
OK: 20250429
OK: 20250430
OK: 20250501
OK: 20250502
OK: 20250504
OK: 20250505
OK: 20250506
OK: 20250507
OK: 20250508
OK: 20250510
OK: 20250511
OK: 20250512
OK: 20250513
OK: 20250516
OK: 20250517
OK: 20250518
OK: 20250519
OK: 20250520
OK: 20250521
OK: 20250523
OK: 20250524
OK: 20250525
OK: 20250526
OK: 20250527
OK: 20250528
OK: 20250530
OK: 20250531
OK: 20250602
OK: 20250603
OK: 20250604
OK: 20250605
OK: 20250606
OK: 20250607
OK: 20250608
OK: 20250609
OK: 20250610
OK: 20250611
OK: 20250612
OK: 20250613
OK: 20250614
OK: 20250615
OK: 20250616
OK: 20250617
OK:

## Extracting all of the zips into their daily folders

In [9]:
ZIP_DIR = BASE483 / "ero_zips_94e09"
EXTRACT_DIR = BASE483 / "ero_tmp"

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

zip_files = sorted(ZIP_DIR.glob("shp_94e_*09.zip"))
for zp in zip_files:
    # pulling out the datetime portion of zip name
    ymd = zp.name.split("_")[2][:8]
    out = EXTRACT_DIR / ymd
    out.mkdir(parents=True, exist_ok=True)

    # skip if already extracted... once again I had a couple already in there
    if list(out.glob("*.shp")):
        continue

    with zipfile.ZipFile(zp, "r") as z:
        z.extractall(out)

print("Done. Extracted folders in:", EXTRACT_DIR)

Zips found: 998
Done. Extracted folders in: /home/scratch/jtoniolo/483/ero_tmp
